# Module 2: Build The Business Graph

This notebook walks through augmenting the metadata graph with business terminology:
1. Load glossary information into Neo4j from CSVs
3. Generate vector embeddings for semantic search
4. Explore the resulting Neo4j graph

**Prerequisites:** Complete `workshop/setup/environment-setup.md` before running this notebook.

## Setup — Imports and Environment

**Sync project dependencies before running this notebook!**

Install libraries to the `uv` managed environment.
```bash
uv sync
```

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

# Verify key variables are set
required_vars = [
    'NEO4J_URI', 'NEO4J_USERNAME', 'NEO4J_PASSWORD', 'NEO4J_DATABASE',
    'OPENAI_API_KEY'
]

print("Environment variable check:")
for var in required_vars:
    val = os.getenv(var)
    status = '✅' if val and val not in ('...', 'neo4j-uri', 'neo4j-password') else '❌'

    print(f"  {status} {var}")

Environment variable check:
  ✅ NEO4J_URI
  ✅ NEO4J_USERNAME
  ✅ NEO4J_PASSWORD
  ✅ NEO4J_DATABASE
  ✅ OPENAI_API_KEY


## Confirm Connections

In [2]:
from neo4j import GraphDatabase
from openai import OpenAI

# Initialize Neo4j driver
neo4j_driver = GraphDatabase.driver(
    uri=os.getenv('NEO4J_URI'),
    auth=(os.getenv('NEO4J_USERNAME'), os.getenv('NEO4J_PASSWORD')),
)
neo4j_database = os.getenv('NEO4J_DATABASE', 'neo4j')

# Verify Neo4j connection
with neo4j_driver.session(database=neo4j_database) as session:
    result = session.run('RETURN 1 AS ping')
    message = '✅' if result.single()['ping'] else '❌'
    print(f"{message} Neo4j driver connected")

# Initialize OpenAI client
embedding_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
print("✅ OpenAI client initialized")

✅ Neo4j driver connected
✅ OpenAI client initialized


## Business Glossary ETL

The `CSVConnector` reads local CSV files and performs the following:
- Extract CSV data 
- Transform extracted data into Pydantic objects
- Load data into Neo4j

In [3]:
from neocarta.connectors.csv import CSVConnector
from pathlib import Path

The CSV files use `ecommerce_db` as a placeholder for the BigQuery project ID.
The utility function below replaces that placeholder with your actual `GCP_PROJECT_ID`
so that `TAGGED_WITH` relationships resolve to the correct `Table` and `Column` nodes.

Note: This creates a temporary directory `/tmp/tmpXXXXXX/` under the OS temp folder with the updated CSV files. This directory is then used for the remainder of the notebook.

In [4]:
import tempfile


def inject_project_id(csv_directory: Path, project_id: str) -> Path:
    """Copy CSV files to a temp dir, replacing the ecommerce_db placeholder."""
    tmp_dir = Path(tempfile.mkdtemp())
    placeholder = 'ecommerce_db'

    for src in csv_directory.glob('*.csv'):
        content = src.read_text()
        (tmp_dir / src.name).write_text(
            content.replace(placeholder, project_id) if placeholder in content else content
        )

    print(f"✅ Injected project_id '{project_id}' into CSV files")
    return tmp_dir

In [5]:
project_id = os.getenv('GCP_PROJECT_ID')
patched_csv_dir = inject_project_id(Path('../datasets/demo/csv/'), project_id)

✅ Injected project_id 'ai-field-alex-g' into CSV files


In [6]:
connector = CSVConnector(
    csv_directory=patched_csv_dir,
    neo4j_driver=neo4j_driver,
    database_name=neo4j_database
)

We can specify the nodes and relationships we would like to ingest with the CSV connector. 
By default, it will attempt to ingest the full graph schema and skip any components that are missing CSV files in the directory.

In [7]:
from neocarta import NodeLabel as nl, RelationshipType as rt

In [8]:
nodes = [
    nl.GLOSSARY,
    nl.CATEGORY,
    nl.BUSINESS_TERM
]
relationships = [
    rt.HAS_CATEGORY, # Glossary -> Category
    rt.HAS_BUSINESS_TERM, # Glossary -> BusinessTerm
    rt.TAGGED_WITH # Business Term -> Table or Column
]

Instead of running each process separately, like we did in the metadata creation process, we will use the `.run()` method to execute the ETL processes in a single call.

In [9]:
connector.run(include_nodes=nodes, include_relationships=relationships)

Extracting metadata from CSV files...
Extracting CSV files from /var/folders/7_/vqs74z3j5hscgzbt0ydbmqyw0000gq/T/tmpd1xnn_w8...
  Extracted 1 rows from glossary_info.csv
  Extracted 9 rows from category_info.csv
  Extracted 76 rows from business_term_info.csv
  Extracted 86 rows from column_term_info.csv
  Extracted 32 rows from table_term_info.csv
Transforming metadata...
Loading metadata into Neo4j...

=== Loading Nodes ===
Loading 1 glossary nodes...
{'_contains_updates': True, 'labels_added': 1, 'nodes_created': 1, 'properties_set': 3}
Loading 9 category nodes...
{'_contains_updates': True, 'labels_added': 9, 'nodes_created': 9, 'properties_set': 27}
Loading 76 business term nodes...
{'_contains_updates': True, 'labels_added': 76, 'nodes_created': 76, 'properties_set': 228}

=== Loading Relationships ===
Loading 9 HAS_CATEGORY relationships...
{'_contains_updates': True, 'relationships_created': 9}
Loading 76 HAS_BUSINESS_TERM relationships...
{'_contains_updates': True, 'relations

Confirm node and relationship counts with the Neo4j driver.

In [10]:
# Verify the node counts in Neo4j
with neo4j_driver.session(database=neo4j_database) as session:
    result = session.run(
        "MATCH (n) RETURN labels(n)[0] AS label, count(n) AS count ORDER BY count ASC"
    )
    print("\nNode counts in Neo4j:")
    for record in result:
        print(f"- {record['label']}: {record['count']}")

# Verify FK relationships
with neo4j_driver.session(database=neo4j_database) as session:
    result = session.run(
        "MATCH ()-[r:REFERENCES]->() RETURN count(r) AS fk_count"
    )
    fk_count = result.single()['fk_count']
    print(f"\nForeign key relationships: {fk_count}")


Node counts in Neo4j:
- Database: 1
- Schema: 1
- __neocarta_graph__: 1
- Glossary: 1
- Category: 9
- Table: 33
- BusinessTerm: 76
- Column: 327
- Value: 2351

Foreign key relationships: 54


## Generate Vector Embeddings

The `OpenAIEmbeddingsConnector` can generate embeddings for nodes containing `description` properties.

Default settings are:
- 768 dimensions
- `text-embedding-3-small` embedding model

Process:
1. Creates a vector index for each node label (if it doesn't exist)
2. Finds all nodes with a `description` field but no `embedding`
3. Calls OpenAI in batches of 100
4. Writes embeddings back to Neo4j

We will generate embeddings for `BusinessTerm` nodes since we will use them as entry points into our semantic graph in addition to the `Schema`, `Table` and `Column` nodes from the previous notebook.

**Note: This will generate an additional 76 embeddings for `BusinessTerm` nodes! This will incur cost!**

In [11]:
from neocarta.enrichment.embeddings import OpenAIEmbeddingsConnector

In [12]:
node_labels = ['BusinessTerm']

openai_embedding_connector = OpenAIEmbeddingsConnector(
    client=embedding_client,
    embedding_model='text-embedding-3-small',
    dimensions=768,
    neo4j_driver=neo4j_driver,
    database_name=neo4j_database,
)

In [13]:
print("Generating embeddings for node labels:", node_labels)

openai_embedding_connector.run(node_labels=node_labels)

print("Embeddings complete!")

Generating embeddings for node labels: ['BusinessTerm']
Processing BusinessTerm nodes...
--------------------------------
Processing batch 1 of 1  
Successful Embeddings : 76
{}
Embeddings complete!


In [14]:
# Verify embedding coverage
with neo4j_driver.session(database=neo4j_database) as session:
    result = session.run("""
        MATCH (n)
        WHERE n.embedding IS NOT NULL OR n.embedding IS NULL
        WITH labels(n)[0] AS label, n
        RETURN label,
               count(n) AS total,
               count(n.embedding) AS with_embedding
        ORDER BY label
    """)
    print("Embedding coverage:")
    for record in result:
        status = '✅' if record['total'] == record['with_embedding'] else '⚠️'
        print(f"  {status} {record['label']}: {record['with_embedding']}/{record['total']} nodes embedded")

Embedding coverage:
  ✅ BusinessTerm: 76/76 nodes embedded
  ⚠️ Category: 0/9 nodes embedded
  ✅ Column: 327/327 nodes embedded
  ⚠️ Database: 0/1 nodes embedded
  ⚠️ Glossary: 0/1 nodes embedded
  ✅ Schema: 1/1 nodes embedded
  ✅ Table: 33/33 nodes embedded
  ⚠️ Value: 0/2351 nodes embedded
  ⚠️ __neocarta_graph__: 0/1 nodes embedded


## Explore the Graph

Use the Neo4j Python driver to run Cypher queries and inspect what was loaded.

Now run some exploratory Cypher queries.

In [43]:
with neo4j_driver.session(database=neo4j_database) as session:
    result = session.run("""
        MATCH (c:Column)-[:TAGGED_WITH]->(b:BusinessTerm)
        RETURN c.name as column,
            b.name as business_term
        ORDER BY column
        LIMIT 5
    """)
    print("Schema Examples:")
    print("-" * 60)
    for record in result.data():
        print(f"{record['column']} -TAGGED_WITH-> {record['business_term']}")

Schema Examples:
------------------------------------------------------------
account_owner_id -TAGGED_WITH-> Account Executive (AE)
account_owner_id -TAGGED_WITH-> Customer Success Manager (CSM)
acquired_date -TAGGED_WITH-> Customer Acquisition Date
allocation_pct -TAGGED_WITH-> Project Assignment
amount_usd -TAGGED_WITH-> Weighted Pipeline


In [44]:
# Show vector indexes
with neo4j_driver.session(database=neo4j_database) as session:
    result = session.run("SHOW VECTOR INDEXES")
    print("Vector Indexes:")
    for record in result:
        print(f"  - {record['name']}: label={record.get('labelsOrTypes', 'N/A')[0]}, state={record['state']}")

Vector Indexes:
  - businessterm_vector_index: label=BusinessTerm, state=ONLINE
  - column_vector_index: label=Column, state=ONLINE
  - schema_vector_index: label=Schema, state=ONLINE
  - table_vector_index: label=Table, state=ONLINE
